# 02 · Serie temporal: ingresos totales

Segunda página de serie temporal. Mismo tratamiento que la de suscriptores —descomposición,
estacionariedad, backtesting walk-forward— pero sobre el ingreso combinado de los tres canales,
y con dos complicaciones que la página anterior no tenía:

1. **"Ingreso" no es una cosa, son tres.** `fct_subscriptions_monthly` ofrece MRR contratado,
   caja cobrada e ingreso reconocido, y no son intercambiables. Elegir mal aquí es peor que
   elegir mal la definición de activo.
2. **El hueco de la migración de pasarela.** Una semana sin datos de pagos a mitad del
   histórico que sólo se ve en una de las tres medidas. Es la imperfección que toca tratar en
   esta página (`docs/data_imperfections.md`).

A cambio, esta página sí tiene grano diario en la parte de tienda, así que aquí la
**estacionalidad semanal** se puede medir de verdad sobre el propio ingreso.

> **Alcance.** "Ingresos totales" son los **cuatro** canales del negocio: suscripción, tienda
> online, boutique y venta de máquinas. La máquina entró al crear el mart `fct_machine_orders`,
> que antes no existía: sin él sólo estaba el agregado por cliente de `dim_customers`, que ni
> tiene grano mensual ni recoge los 616 pedidos de boutique sin fidelización. Con el mart, la
> máquina resulta pesar **840.859 €, un 21,5% del ingreso** — bastante más de lo que sugería el
> agregado por cliente.

In [1]:
import json
import sys
import warnings
from datetime import datetime, timezone
from pathlib import Path

import duckdb
import numpy as np
import pandas as pd
import plotly.graph_objects as go
from plotly.subplots import make_subplots

PROJECT_ROOT = Path.cwd()
while not (PROJECT_ROOT / "data" / "warehouse.duckdb").exists() and PROJECT_ROOT != PROJECT_ROOT.parent:
    PROJECT_ROOT = PROJECT_ROOT.parent
sys.path.insert(0, str(PROJECT_ROOT / "analysis"))

import utils_timeseries as ts
import utils_subscriptions as us

DB_PATH = PROJECT_ROOT / "data" / "warehouse.duckdb"
OUTPUT_PATH = PROJECT_ROOT / "analysis" / "outputs" / "ingresos.json"
OUTPUT_PATH.parent.mkdir(parents=True, exist_ok=True)

# --- Parámetros del análisis ---
FORECAST_HORIZON = 6
BACKTEST_FOLDS = 5
SEASON_LENGTH = 12
ALPHA = 0.05
INTERVAL_LEVEL = 0.80
# Modelo airline (0,1,1)(0,1,1)12: una diferencia regular, una estacional y una media móvil en
# cada una. Es la referencia clásica para series mensuales con tendencia y estacionalidad, y se
# elige *a priori* — no por ser la que mejor sale en el backtesting.
SARIMA_ORDER = (0, 1, 1)
SARIMA_SEASONAL = (0, 1, 1, SEASON_LENGTH)

C_BLUE, C_ORANGE, C_AQUA, C_YELLOW = "#2a78d6", "#eb6834", "#1baf7a", "#eda100"
C_VIOLET, C_RED = "#4a3aa7", "#e34948"
C_GRID, C_INK, C_MUTED = "#e6e6e3", "#0b0b0b", "#52514e"
CHANNEL_COLOR = {"suscripcion": C_BLUE, "online": C_ORANGE, "tienda": C_AQUA,
                 "maquina": C_YELLOW}
CHANNEL_LABEL = {"suscripcion": "Suscripción", "online": "Tienda online",
                 "tienda": "Boutique", "maquina": "Máquinas"}

PLOT_LAYOUT = dict(
    template="plotly_white", height=420,
    margin=dict(l=70, r=30, t=60, b=50),
    font=dict(color=C_INK, size=12),
    legend=dict(orientation="h", yanchor="bottom", y=1.02, x=0),
    xaxis=dict(gridcolor=C_GRID), yaxis=dict(gridcolor=C_GRID),
    hovermode="x unified",
)

pd.set_option("display.width", 200)
pd.set_option("display.max_columns", 40)
print("proyecto:", PROJECT_ROOT.name, "| duckdb:", DB_PATH.exists())

proyecto: capsule-club-analytics | duckdb: True


## 1. Las tres medidas de ingreso, y una trampa de grano

`dbt_project/README.md` avisa de las tres medidas de la suscripción:

| Columna | Qué mide | Cuándo usarla |
|---|---|---|
| `contracted_mrr_eur` | Tarifa vigente de lo activo y sin pausa. | Señal limpia de la base contratada. No acusa problemas de cobro. |
| `collected_eur` | Caja realmente cobrada ese mes. | Tesorería. **Sí** acusa el hueco de la pasarela. |
| `recognized_revenue_eur` | El cobro repartido entre los meses de su ciclo. | Ingreso económico. Es la que se usa como serie protagonista. |

Se elige **`recognized_revenue_eur`** porque es la única que representa ingreso devengado: los
planes anual y trimestral cobran de golpe, y usar la caja metería picos que no corresponden al
mes en que se presta el servicio.

Y en `fct_shop_orders` hay una trampa de grano documentada: el mart está a nivel de **línea de
pedido**, y `order_total_eur` viene repetido en cada línea. Sumarlo sin deduplicar infla el
ingreso de tienda.

In [2]:
con = duckdb.connect(str(DB_PATH), read_only=True)

facts = con.sql("select * from fct_subscriptions_monthly").df()

subs_monthly = con.sql("""
    select month_start,
           sum(contracted_mrr_eur)      as mrr_eur,
           sum(collected_eur)           as collected_eur,
           sum(recognized_revenue_eur)  as recognized_eur,
           sum(payments_collected)      as payments,
           sum(is_active_net_of_pauses::int) as active
    from fct_subscriptions_monthly
    group by 1
""").df()

shop_daily = con.sql("""
    select order_date, channel, sum(line_amount_eur) as revenue_eur
    from fct_shop_orders
    group by 1, 2
""").df()

machine_monthly = con.sql("""
    select order_month, sum(price_paid_eur) as revenue_eur, count(*) as orders
    from fct_machine_orders
    group by 1
""").df()

# La trampa de grano, medida en vez de contada.
grain_check = con.sql("""
    select round(sum(line_amount_eur))                                       as suma_lineas,
           (select round(sum(order_total_eur))
              from (select distinct order_id, order_total_eur from fct_shop_orders)) as suma_dedup,
           round(sum(order_total_eur))                                       as suma_naive
    from fct_shop_orders
""").df()
con.close()

subs_monthly["month_start"] = pd.to_datetime(subs_monthly["month_start"])
shop_daily["order_date"] = pd.to_datetime(shop_daily["order_date"])
machine_monthly["order_month"] = pd.to_datetime(machine_monthly["order_month"])
facts["month_start"] = pd.to_datetime(facts["month_start"])
facts["cohort_month"] = pd.to_datetime(facts["cohort_month"])
for col in ["is_active_eom", "is_paused", "is_active_net_of_pauses"]:
    facts[col] = facts[col].astype(bool)

print(grain_check.to_string(index=False))
inflacion = grain_check.suma_naive.iloc[0] / grain_check.suma_lineas.iloc[0]
print()
print(f"Sumar order_total_eur sin deduplicar infla el ingreso de tienda x{inflacion:.2f}.")
print("sum(line_amount_eur) y el total deduplicado por pedido coinciden:",
      bool(grain_check.suma_lineas.iloc[0] == grain_check.suma_dedup.iloc[0]))

 suma_lineas  suma_dedup  suma_naive
   1758474.0   1758474.0   4237572.0

Sumar order_total_eur sin deduplicar infla el ingreso de tienda x2.41.
sum(line_amount_eur) y el total deduplicado por pedido coinciden: True


## 2. El hueco de la migración de pasarela

`docs/data_imperfections.md` documenta una migración de pasarela a mitad del histórico que deja
**una semana sin datos de pagos**, y añade la pista de cómo encontrarla: afecta sólo a los
cobros, no a los eventos de suscripción, así que *el MRR contratado no debe acusarla y la caja
sí*.

Para buscarla no vale mirar el importe absoluto —la serie crece mucho— sino **normalizar por
suscriptor activo**. Y hay un distractor que conviene resolver antes: febrero siempre sale bajo
en número de pagos simplemente por tener menos días.

In [3]:
gap = subs_monthly.set_index("month_start").sort_index()
gap["collected_per_active"] = gap.collected_eur / gap.active
gap["mrr_per_active"] = gap.mrr_eur / gap.active
gap["payments_per_active"] = gap.payments / gap.active
gap["month"] = gap.index.month

# Primer intento: pagos por activo. Funciona a medias, porque febrero sale siempre
# bajo por tener menos días y ensucia el ranking.
febreros = gap[gap.month == 2].payments_per_active
resto = gap[gap.month != 2].payments_per_active
print(f"Pagos por activo: media de febreros = {febreros.mean():.3f} · "
      f"resto de meses = {resto.mean():.3f}")
print("Febrero sale bajo por ser mes corto, no por un incidente: hace de distractor.")
print()
print("Meses con menor ratio de pagos por activo (ranking sucio):")
print(gap.nsmallest(5, "payments_per_active")[["payments_per_active"]].round(3).to_string())

# Señal limpia: euros cobrados por cada euro de MRR contratado. El denominador es la
# tarifa vigente, que ni crece con el negocio ni depende de los días del mes, así que
# el ratio aísla exactamente lo que falla — el registro del cobro.
gap["cash_ratio"] = gap.collected_eur / gap.mrr_eur
mediana = gap.cash_ratio.median()
print()
print(f"Ratio caja / MRR contratado — mediana del histórico: {mediana:.3f}")
print("Los cinco meses más bajos:")
print(gap.nsmallest(5, "cash_ratio")[["cash_ratio", "collected_per_active",
                                      "mrr_per_active"]].round(3).to_string())

GAP_MONTH = gap.cash_ratio.idxmin()
caida = 1 - gap.loc[GAP_MONTH, "cash_ratio"] / mediana
print()
print(f"Mínimo absoluto: {GAP_MONTH:%Y-%m} con {gap.loc[GAP_MONTH, 'cash_ratio']:.3f}, "
      f"un {caida:.1%} por debajo de la mediana.")

Pagos por activo: media de febreros = 0.589 · resto de meses = 0.674
Febrero sale bajo por ser mes corto, no por un incidente: hace de distractor.

Meses con menor ratio de pagos por activo (ranking sucio):
             payments_per_active
month_start                     
2023-09-01                 0.531
2024-02-01                 0.559
2025-03-01                 0.573
2025-09-01                 0.582
2026-02-01                 0.604

Ratio caja / MRR contratado — mediana del histórico: 1.020
Los cinco meses más bajos:
             cash_ratio  collected_per_active  mrr_per_active
month_start                                                  
2025-03-01        0.819                24.521          29.951
2025-09-01        0.839                25.153          29.976
2024-08-01        0.851                25.401          29.853
2026-02-01        0.895                26.839          29.974
2025-02-01        0.908                27.092          29.831

Mínimo absoluto: 2025-03 con 0.819, un 1

In [4]:
fig = make_subplots(rows=2, cols=1, shared_xaxes=True, vertical_spacing=0.09,
                    subplot_titles=("Caja cobrada por suscriptor activo (€)",
                                    "MRR contratado por suscriptor activo (€)"))
fig.add_trace(go.Scatter(x=gap.index, y=gap.collected_per_active, mode="lines",
                         line=dict(color=C_ORANGE, width=2), showlegend=False), row=1, col=1)
fig.add_trace(go.Scatter(x=gap.index, y=gap.mrr_per_active, mode="lines",
                         line=dict(color=C_BLUE, width=2), showlegend=False), row=2, col=1)
for r in (1, 2):
    fig.add_vrect(x0=GAP_MONTH, x1=GAP_MONTH + pd.offsets.MonthEnd(0),
                  fillcolor=C_RED, opacity=0.12, line_width=0, row=r, col=1)
fig.add_annotation(x=GAP_MONTH, yref="y domain", y=0.06, row=1, col=1, xanchor="left",
                   text="  migración de pasarela", showarrow=False,
                   font=dict(color=C_MUTED, size=11))
layout = {k: v for k, v in PLOT_LAYOUT.items() if k not in ("height", "xaxis", "yaxis", "legend")}
fig.update_layout(**layout, height=560, showlegend=False,
                  title="La misma semana, dos medidas: sólo la caja tiene el agujero")
fig.update_xaxes(gridcolor=C_GRID); fig.update_yaxes(gridcolor=C_GRID)
fig.show()

vecinos = gap.loc[GAP_MONTH - pd.DateOffset(months=2):GAP_MONTH + pd.DateOffset(months=2),
                  ["collected_per_active", "mrr_per_active", "cash_ratio"]]
print(vecinos.round(3).to_string())
print()
MESES_ES = ["enero", "febrero", "marzo", "abril", "mayo", "junio",
            "julio", "agosto", "septiembre", "octubre", "noviembre", "diciembre"]
gap_label = f"{MESES_ES[GAP_MONTH.month - 1]} de {GAP_MONTH.year}"
print(f"{gap_label} cobra {gap.loc[GAP_MONTH, 'collected_per_active']:.2f} € por activo "
      f"frente a los {vecinos.collected_per_active.drop(GAP_MONTH).mean():.2f} € de sus vecinos, "
      f"con el MRR contratado plano en {vecinos.mrr_per_active.mean():.2f} €.")

             collected_per_active  mrr_per_active  cash_ratio
month_start                                                  
2025-01-01                 30.875          29.843       1.035
2025-02-01                 27.092          29.831       0.908
2025-03-01                 24.521          29.951       0.819
2025-04-01                 29.163          29.952       0.974
2025-05-01                 30.776          29.902       1.029

marzo de 2025 cobra 24.52 € por activo frente a los 29.48 € de sus vecinos, con el MRR contratado plano en 29.90 €.


**Encontrado: marzo de 2025.** El ratio caja / MRR contratado cae a **0,819 cuando la mediana
del histórico es 1,02**, un 20% por debajo, y es el mínimo absoluto de los 36 meses. El MRR
contratado, mientras tanto, no se mueve: 29,95 € por activo, igual que en febrero y abril.

El ratio es mejor herramienta que el importe cobrado por activo porque su denominador —la
tarifa vigente— ni crece con el negocio ni depende de los días del mes. Eso deja fuera los dos
distractores: la tendencia y los febreros cortos.

Es exactamente la firma que describe el catálogo: no se perdió negocio, se perdieron *registros
de cobro*. El segundo mes más bajo (septiembre de 2025, 0,839) no es otro incidente: es el
rebote de las pausas de verano, cuando vuelve a facturarse gente que llevaba meses sin cobrar.

La consecuencia práctica es la razón por la que esta página usa ingreso reconocido y no caja: si
se pronosticara sobre `collected_eur`, el modelo aprendería un bache que no volverá a pasar.

## 3. Ingreso total y desglose por canal

In [5]:
recognized = ts.build_series(subs_monthly, "month_start", "recognized_eur", freq="MS")
online = ts.build_series(shop_daily[shop_daily.channel == "online"], "order_date",
                         "revenue_eur", freq="MS")
store = ts.build_series(shop_daily[shop_daily.channel == "store"], "order_date",
                        "revenue_eur", freq="MS")

machines = ts.build_series(machine_monthly, "order_month", "revenue_eur", freq="MS")

channels = {"suscripcion": recognized, "online": online, "tienda": store, "maquina": machines}
total_revenue = sum(channels.values())
total_revenue.name = "total_eur"

mix = pd.DataFrame(channels).assign(total=total_revenue)
share = (mix[list(channels)].sum() / mix.total.sum() * 100).round(1)
print("Reparto del ingreso del histórico (%):")
print(share.to_string())
print()
print(f"Ingreso total del histórico: {total_revenue.sum():,.0f} €")
print(f"Serie: {len(total_revenue)} meses · {total_revenue.iloc[0]:,.0f} € → "
      f"{total_revenue.iloc[-1]:,.0f} €")
(mix / 1000).round(1).tail(8)

Reparto del ingreso del histórico (%):
suscripcion    33.5
online         25.2
tienda         19.8
maquina        21.5

Ingreso total del histórico: 3,907,789 €
Serie: 36 meses · 10,844 € → 191,770 €


,suscripcion,online,tienda,maquina,total
2026-01-01,61.3,64.7,39.4,38.6,203.9
2026-02-01,61.0,56.4,35.7,31.5,184.5
2026-03-01,75.6,67.5,39.0,33.9,215.9
2026-04-01,72.3,63.3,38.4,39.7,213.7
2026-05-01,79.2,65.7,39.8,44.7,229.4
2026-06-01,77.4,63.3,33.4,41.4,215.6
2026-07-01,77.5,52.8,29.3,37.4,197.0
2026-08-01,69.3,54.5,31.7,36.3,191.8


In [6]:
fig = go.Figure()
for name, s in channels.items():
    fig.add_trace(go.Scatter(x=s.index, y=s.to_numpy(), name=CHANNEL_LABEL[name],
                             mode="lines", stackgroup="one",
                             line=dict(color=CHANNEL_COLOR[name], width=0.5)))
fig.update_layout(**{**PLOT_LAYOUT, "height": 460},
                  title="Ingreso mensual por canal (apilado)", yaxis_title="€ / mes")
fig.show()

# Peso relativo: ¿está cambiando el mix?
mix_pct = mix[list(channels)].div(mix.total, axis=0) * 100
fig = go.Figure()
for name in channels:
    fig.add_trace(go.Scatter(x=mix_pct.index, y=mix_pct[name], name=CHANNEL_LABEL[name],
                             mode="lines", line=dict(color=CHANNEL_COLOR[name], width=2)))
fig.update_layout(**PLOT_LAYOUT, title="Peso de cada canal sobre el ingreso total (%)",
                  yaxis_title="% del total")
fig.show()
print(mix_pct.round(1).iloc[[0, 11, 23, 35]].to_string())

            suscripcion  online  tienda  maquina
2023-09-01          3.2     1.8    41.1     53.9
2024-08-01         42.8    11.9    21.2     24.1
2025-08-01         42.5    19.4    15.8     22.4
2026-08-01         36.1    28.4    16.5     18.9


El mix se mueve de forma clara: la suscripción arranca siendo casi todo el ingreso recurrente y
va cediendo peso a la tienda a medida que el catálogo y las boutiques maduran. Ninguno de los
cuatro canales domina: 33,5% suscripción, 25,2% tienda online, 21,5% máquinas y 19,8% boutique.

Para el forecast esto importa porque **los cuatro canales no tienen la misma estacionalidad**,
así que el agregado hereda una mezcla que además cambia con el tiempo.

In [7]:
decompositions = {name: ts.stl_decompose(s, period=SEASON_LENGTH, robust=True)
                  for name, s in channels.items()}
decomposition = ts.stl_decompose(total_revenue, period=SEASON_LENGTH, robust=True)

strength = pd.DataFrame({
    "F_tendencia": {**{n: d.trend_strength for n, d in decompositions.items()},
                    "TOTAL": decomposition.trend_strength},
    "F_estacional": {**{n: d.seasonal_strength[SEASON_LENGTH] for n, d in decompositions.items()},
                     "TOTAL": decomposition.seasonal_strength[SEASON_LENGTH]},
}).round(3)
print(strength.to_string())

components = decomposition.to_frame()
fig = make_subplots(rows=4, cols=1, shared_xaxes=True, vertical_spacing=0.05,
                    subplot_titles=("Observado", "Tendencia", "Estacionalidad (12m)", "Residuo"))
for row, (col, color) in enumerate([("observed", C_BLUE), ("trend", C_ORANGE),
                                    (f"seasonal_{SEASON_LENGTH}", C_AQUA), ("resid", C_MUTED)], start=1):
    fig.add_trace(go.Scatter(x=components.index, y=components[col].to_numpy(), mode="lines",
                             line=dict(color=color, width=2), showlegend=False), row=row, col=1)
fig.add_hline(y=0, line=dict(color=C_GRID, width=1), row=4, col=1)
layout = {k: v for k, v in PLOT_LAYOUT.items() if k not in ("height", "xaxis", "yaxis", "legend")}
fig.update_layout(**layout, height=760, showlegend=False,
                  title="Descomposición STL del ingreso total")
fig.update_xaxes(gridcolor=C_GRID); fig.update_yaxes(gridcolor=C_GRID)
fig.show()

             F_tendencia  F_estacional
suscripcion        0.995         0.774
online             0.986         0.855
tienda             0.989         0.942
maquina            0.982         0.835
TOTAL              0.992         0.853


In [8]:
# Perfil estacional de cada canal, en % sobre su propia tendencia, para poder compararlos
# pese a que sus niveles son muy distintos.
#
# Dos precauciones que cambian el resultado por completo: se usan sólo los últimos 24 meses y
# se toma la mediana. Durante el año de arranque la tendencia vale unos pocos miles de euros,
# así que dividir por ella dispara el porcentaje —salían diciembres de +171%— y la media
# arrastra esos valores al perfil. Con el tramo maduro y la mediana, el perfil es interpretable.
PROFILE_MONTHS = 24
profiles = {}
for name, d in decompositions.items():
    rel = ((d.seasonal[SEASON_LENGTH] / d.trend) * 100).iloc[-PROFILE_MONTHS:]
    profiles[name] = rel.groupby(rel.index.month).median()
profile_frame = pd.DataFrame(profiles)
profile_frame.index = ["ene","feb","mar","abr","may","jun","jul","ago","sep","oct","nov","dic"]

fig = go.Figure()
for name in channels:
    fig.add_trace(go.Bar(x=profile_frame.index, y=profile_frame[name],
                         name=CHANNEL_LABEL[name], marker_color=CHANNEL_COLOR[name]))
fig.update_layout(**{**PLOT_LAYOUT, "hovermode": "closest"}, barmode="group",
                  title="Perfil estacional por canal (% sobre tendencia, últimos 24 meses)",
                  yaxis_title="% sobre tendencia")
fig.show()
print(profile_frame.round(1).to_string())

     suscripcion  online  tienda  maquina
ene          0.8    21.5    12.9      5.7
feb         -3.9    -3.2    -1.3    -14.8
mar         15.1    11.3     2.6    -10.4
abr          6.6    -2.2    -1.5     -1.7
may         11.9    -3.2    -1.2      5.1
jun          5.7   -11.4   -22.0     -7.2
jul          2.5   -32.3   -33.0    -15.1
ago        -12.1   -33.2   -30.7    -23.7
sep        -16.3     1.9    -0.5      9.0
oct         -3.1    25.2    27.7     24.8
nov         -6.4    29.7    31.1     16.3
dic          2.1    23.0    23.6     18.6


Los perfiles confirman que **la boutique es el canal más estacional** (F_S = 0,94) y la
suscripción el menos (0,77), que es justo lo que cabe esperar: un contrato recurrente amortigua
el ciclo, y una tienda física lo sufre entero.

Pero lo interesante es que **no comparten el calendario**. Los dos canales de tienda tienen el
mismo ciclo de café —techo en octubre-noviembre y suelo en julio-agosto—, mientras que la
suscripción hace máximo en marzo y mínimo en agosto-septiembre, arrastrada por las pausas de
verano y su rebote.

Es decir: el agregado mezcla estacionalidades desfasadas, y eso es justo lo que el modelo
top-down tiene que aprender de una sola serie. Las comparaciones de más abajo —bottom-up por
canal y encadenado con el modelo de cohortes— existen para comprobar si le compensa.

## 4. Grano diario: la estacionalidad semanal

Aquí sí se puede medir, porque `fct_shop_orders` tiene fecha de pedido. Se descompone con MSTL
para separar el ciclo semanal del anual en la misma pasada.

Ojo al alcance: esto es **sólo la parte de tienda** (online + boutique, el 45% del ingreso). La
suscripción se factura por ciclos mensuales y no tiene grano diario en el mart; la máquina sí lo
tendría, pero es una compra de muy baja frecuencia y su serie diaria es casi toda ceros, así que
tampoco entra.

In [9]:
HISTORY_END = total_revenue.index.max() + pd.offsets.MonthEnd(0)
shop_total_daily = ts.build_series(
    shop_daily.groupby("order_date", as_index=False).revenue_eur.sum(),
    "order_date", "revenue_eur", freq="D",
    start=total_revenue.index.min(), end=HISTORY_END)

daily_decomposition = ts.mstl_decompose(
    shop_total_daily, periods=ts.infer_seasonal_periods(shop_total_daily))

print(f"{len(shop_total_daily)} días · {shop_total_daily.sum():,.0f} € · "
      f"media {shop_total_daily.mean():,.0f} €/día")
print(f"periodos modelados: {daily_decomposition.periods}")
for period, value in daily_decomposition.seasonal_strength.items():
    print(f"  fuerza estacional {'semanal' if period == 7 else 'anual':8s} "
          f"(m={period:3d}) = {value:.3f}")
print(f"  fuerza de la tendencia                  = {daily_decomposition.trend_strength:.3f}")

DAYS = ["lun", "mar", "mié", "jue", "vie", "sáb", "dom"]
by_dow = {}
for name, label in [("online", "online"), ("store", "tienda")]:
    s = ts.build_series(shop_daily[shop_daily.channel == name], "order_date", "revenue_eur",
                        freq="D", start=total_revenue.index.min(), end=HISTORY_END)
    by_dow[label] = s.groupby(s.index.dayofweek).mean()
by_dow["total"] = shop_total_daily.groupby(shop_total_daily.index.dayofweek).mean()
dow_frame = pd.DataFrame(by_dow); dow_frame.index = DAYS
print()
print("Ingreso medio por día de la semana (€):")
print(dow_frame.round(0).to_string())
weekend_ratio = float(dow_frame.total.iloc[5:].mean() / dow_frame.total.iloc[:5].mean())
print()
print(f"Fin de semana frente a laborable: {weekend_ratio:.3f} "
      f"({(1 - weekend_ratio) * 100:.1f}% menos de ingreso)")

1096 días · 1,758,474 € · media 1,604 €/día
periodos modelados: (7, 365)
  fuerza estacional semanal  (m=  7) = 0.651
  fuerza estacional anual    (m=365) = 0.777
  fuerza de la tendencia                  = 0.959

Ingreso medio por día de la semana (€):
     online  tienda   total
lun  1028.0   803.0  1832.0
mar   999.0   803.0  1802.0
mié  1001.0   775.0  1776.0
jue   956.0   775.0  1731.0
vie   916.0   711.0  1628.0
sáb   762.0   575.0  1337.0
dom   627.0   502.0  1129.0

Fin de semana frente a laborable: 0.703 (29.7% menos de ingreso)


In [10]:
fig = make_subplots(rows=2, cols=1, vertical_spacing=0.16,
                    subplot_titles=("Componente semanal del ingreso de tienda (€ sobre la media)",
                                    "Componente anual (€ sobre la media)"))
weekly = daily_decomposition.seasonal[7]
weekly_profile = weekly.groupby(weekly.index.dayofweek).mean()
fig.add_trace(go.Bar(x=DAYS, y=weekly_profile.to_numpy(),
                     marker_color=[C_RED if v < 0 else C_BLUE for v in weekly_profile],
                     text=[f"{v:+.0f}" for v in weekly_profile], textposition="outside",
                     showlegend=False), row=1, col=1)
annual = daily_decomposition.seasonal[365]
fig.add_trace(go.Scatter(x=annual.index, y=annual.to_numpy(), mode="lines",
                         line=dict(color=C_AQUA, width=2), showlegend=False), row=2, col=1)
fig.add_hline(y=0, line=dict(color=C_GRID, width=1), row=2, col=1)
layout = {k: v for k, v in PLOT_LAYOUT.items()
          if k not in ("height", "xaxis", "yaxis", "legend", "hovermode")}
fig.update_layout(**layout, height=640, showlegend=False, hovermode="closest",
                  title="Ingreso de tienda: estacionalidad semanal y anual con MSTL")
fig.update_xaxes(gridcolor=C_GRID); fig.update_yaxes(gridcolor=C_GRID)
fig.show()

Esta es la diferencia grande con la página anterior. Allí la estacionalidad semanal de las altas
era real pero modesta (F_S ≈ 0,21); **aquí pesa tres veces más (F_S = 0,65)** y tiene una forma
muy marcada: el ingreso cae de forma monótona de lunes a domingo, y el fin de semana factura un
30% menos que un día laborable. Online y boutique se comportan casi igual (ratio 0,71 y 0,70),
lo que descarta que sea un efecto de horario de tienda física.

Para el forecast mensual esto no cambia nada —al agregar a mes el ciclo semanal se promedia—
pero sí cambia la planificación operativa: el reparto de carga de almacén y de atención al
cliente no es plano dentro de la semana.

## 5. Estacionariedad

In [11]:
MAXLAG = int(np.ceil(4 * (len(total_revenue) / 100) ** (2 / 9)))
stat_kwargs = dict(season_length=SEASON_LENGTH, alpha=ALPHA, regression="ct", maxlag=MAXLAG)

report_level = ts.stationarity_report(total_revenue, **stat_kwargs)
report_log = ts.stationarity_report(np.log(total_revenue), **stat_kwargs)
print(f"maxlag = {MAXLAG} (acotado: con {len(total_revenue)} puntos la regla por defecto "
      f"pediría {int(np.ceil(12 * (len(total_revenue) / 100) ** 0.25))})")
print()
print("Serie en nivel :", report_level.summary())
print("Serie en log   :", report_log.summary())
print()
for name, s in channels.items():
    r = ts.stationarity_report(np.log(s.clip(lower=1)), **stat_kwargs)
    print(f"  {CHANNEL_LABEL[name]:14s} (log): d={r.n_diffs}, D={r.n_seasonal_diffs} · "
          f"ADF p={r.adf.p_value:.4f}")

maxlag = 4 (acotado: con 36 puntos la regla por defecto pediría 10)

Serie en nivel : discrepancia: ADF no rechaza la raíz unitaria pero KPSS no rechaza la estacionariedad | ADF p=0.2839 | KPSS p=0.1000 (recortado) | d=1, D=1 (m=12)
Serie en log   : no estacionaria (ADF y KPSS coinciden) | ADF p=0.3734 | KPSS p=0.0124 | d=0, D=1 (m=12)

  Suscripción    (log): d=0, D=0 · ADF p=0.0000
  Tienda online  (log): d=0, D=1 · ADF p=0.0277
  Boutique       (log): d=0, D=1 · ADF p=0.1873


  Máquinas       (log): d=1, D=1 · ADF p=0.0118


La conclusión es la misma que en la página anterior y por las mismas razones: serie corta,
fuertemente tendencial, los dos contrastes con poca potencia. Se modela en **logaritmos** —el
ingreso crece de forma multiplicativa y la varianza sube con el nivel— y con una diferencia
regular y una estacional.

## 6. Modelos y backtesting

Tres enfoques, con la misma firma `forecast_fn(train, horizon)` y por tanto las mismas métricas
y los mismos pliegues:

| Modelo | Qué hace |
|---|---|
| `naive_estacional` | Repite el mismo mes del año anterior. Suelo de referencia. |
| `sarima_airline` (top-down) | Un SARIMA(0,1,1)(0,1,1)₁₂ sobre el log del **agregado**. |
| `bottom_up_canal` | Un SARIMA igual por **cada canal**, y se suman los cuatro. |
| `encadenado_cohortes` | La suscripción se pronostica como *activos × ingreso por activo*, usando el modelo de cohortes de la página anterior; los otros tres canales, con SARIMA. |

Dos preguntas concretas: dado que los canales tienen estacionalidades distintas, **¿compensa
modelarlos por separado?** Y dado que el canal de suscripción es contractual y se puede
pronosticar con cohortes, **¿mejora eso el agregado?**

In [12]:
def sarima_forecast(train, horizon, order=SARIMA_ORDER, seasonal=SARIMA_SEASONAL,
                    level=INTERVAL_LEVEL):
    """SARIMA sobre log. Devuelve punto e intervalo en la escala original."""
    from statsmodels.tsa.statespace.sarimax import SARIMAX
    with warnings.catch_warnings():
        warnings.simplefilter("ignore")
        fitted = SARIMAX(np.log(train), order=order, seasonal_order=seasonal,
                         enforce_stationarity=False, enforce_invertibility=False).fit(disp=False)
        forecast = fitted.get_forecast(horizon)
        conf = forecast.conf_int(alpha=1 - level)
    return pd.DataFrame({
        "yhat": np.exp(forecast.predicted_mean.to_numpy()),
        "yhat_lower": np.exp(conf.iloc[:, 0].to_numpy()),
        "yhat_upper": np.exp(conf.iloc[:, 1].to_numpy()),
    })
sarima_forecast.__name__ = "sarima_airline"


# Base del encadenado: activos a cierre de mes (sin descontar pausas) y el ingreso
# reconocido por cada uno. La pausa no se descuenta aquí porque el ingreso reconocido
# reparte el cobro entre los meses del ciclo, así que un mes pausado sigue devengando.
active_eom = ts.build_series(
    facts.assign(v=facts.is_active_eom.astype(int)), "month_start", "v", freq="MS")
revenue_per_active = recognized / active_eom


def chained_forecast(train, horizon):
    """Suscripción = activos (modelo de cohortes) x ingreso por activo; resto, SARIMA.

    Es el encadenado con la página 2. La mecánica de cohortes vive en
    `analysis/utils_subscriptions.py`, así que este notebook no depende del JSON de
    salida del anterior: no hay orden de ejecución impuesto, sólo import.
    """
    cutoff = train.index[-1]
    subscribers = us.forecast_active_subscribers(facts, cutoff, horizon, apply_pause=False)
    arpu = sarima_forecast(revenue_per_active.reindex(train.index), horizon)["yhat"].to_numpy()
    total = subscribers * arpu
    for name in ("online", "tienda", "maquina"):
        total = total + sarima_forecast(channels[name].reindex(train.index),
                                        horizon)["yhat"].to_numpy()
    return total
chained_forecast.__name__ = "encadenado_cohortes"


def bottom_up_forecast(train, horizon):
    """Suma de un SARIMA por canal, recortando cada canal al mismo tramo que el agregado.

    El recorte por `train.index` es lo que evita la fuga de información: cada canal ve
    exactamente el mismo histórico que vería el modelo top-down en ese pliegue.
    """
    parts = [sarima_forecast(s.reindex(train.index), horizon)["yhat"].to_numpy()
             for s in channels.values()]
    return np.sum(parts, axis=0)
bottom_up_forecast.__name__ = "bottom_up_canal"


models = {
    "naive_estacional": ts.make_seasonal_naive(SEASON_LENGTH),
    "sarima_airline": sarima_forecast,
    "bottom_up_canal": bottom_up_forecast,
    "encadenado_cohortes": chained_forecast,
}
backtests = {
    name: ts.walk_forward_backtest(total_revenue, fn, horizon=FORECAST_HORIZON,
                                   n_folds=BACKTEST_FOLDS, step=1, season_length=SEASON_LENGTH,
                                   model_name=name, on_error="skip")
    for name, fn in models.items()
}
leaderboard = ts.compare_backtests(backtests)
leaderboard[["model", "n_folds", "mae", "mape", "smape", "mase", "bias", "coverage",
             "failed_folds"]].round(3)

,model,n_folds,mae,mape,smape,mase,bias,coverage,failed_folds
0,sarima_airline,5,23557.398,11.320,10.800,0.324,12259.077,70.0,0
1,encadenado_cohortes,5,24168.529,11.661,10.871,0.331,21368.466,NaN,0
2,bottom_up_canal,5,27108.365,13.078,12.054,0.374,24218.202,NaN,0
3,naive_estacional,5,105564.619,50.667,68.059,1.445,-105564.619,NaN,0


In [13]:
for name, result in backtests.items():
    print(f"{name:18s} MAPE por horizonte:",
          [round(r["mape"], 1) for r in result.metrics_by_horizon.to_dict("records")])

# Robustez: horizonte más corto y más pliegues.
short = {name: ts.walk_forward_backtest(total_revenue, fn, horizon=3, n_folds=9, step=1,
                                        season_length=SEASON_LENGTH, model_name=name,
                                        on_error="skip")
         for name, fn in models.items()}
print()
print("Horizonte 3 meses, 9 pliegues:")
print(ts.compare_backtests(short)[["model", "mae", "mape", "mase", "bias"]]
      .round(3).to_string(index=False))

naive_estacional   MAPE por horizonte: [54.9, 52.5, 51.1, 49.4, 49.1, 47.0]
sarima_airline     MAPE por horizonte: [12.4, 3.7, 13.7, 11.4, 11.1, 15.6]
bottom_up_canal    MAPE por horizonte: [8.5, 8.8, 13.5, 15.2, 14.2, 18.2]
encadenado_cohortes MAPE por horizonte: [8.4, 8.3, 12.2, 13.7, 12.0, 15.4]



Horizonte 3 meses, 9 pliegues:
              model        mae   mape  mase        bias
encadenado_cohortes  16334.449  7.967 0.220   12676.204
     sarima_airline  18213.157  8.858 0.244    8263.349
    bottom_up_canal  20027.603  9.723 0.268   16382.970
   naive_estacional 105772.101 51.426 1.427 -105772.101


In [14]:
# ¿Qué canal es más difícil de pronosticar?
per_channel = {
    name: ts.walk_forward_backtest(s, sarima_forecast, horizon=FORECAST_HORIZON,
                                   n_folds=BACKTEST_FOLDS, step=1, season_length=SEASON_LENGTH,
                                   model_name=name, on_error="skip")
    for name, s in channels.items()
}
per_channel_table = ts.compare_backtests(per_channel)[
    ["model", "mae", "mape", "mase", "bias", "coverage"]].round(3)
print(per_channel_table.to_string(index=False))

fig = go.Figure()
for name in channels:
    r = per_channel[name]
    fig.add_trace(go.Scatter(x=r.metrics_by_horizon.h, y=r.metrics_by_horizon.mape,
                             name=CHANNEL_LABEL[name], mode="lines+markers",
                             line=dict(color=CHANNEL_COLOR[name], width=2), marker=dict(size=8)))
fig.update_layout(**{**PLOT_LAYOUT, "hovermode": "closest"},
                  title="Error del forecast por canal y horizonte",
                  xaxis_title="Meses de horizonte", yaxis_title="MAPE (%)")
fig.show()

      model       mae   mape  mase     bias  coverage
suscripcion  7154.478 10.100 0.285 4199.650    43.333
     tienda  6577.336 17.458 0.504 4071.343    50.000
    maquina  9086.553 24.560 0.685 8172.155    53.333
     online 17321.681 27.581 0.804 7775.055    83.333


**Tres lecturas del backtesting.**

1. **Desagregar no compensa en el agregado.** Ni el `bottom_up` (MASE 0,375) ni el
   `encadenado_cohortes` (0,329) baten al SARIMA directo sobre el total (0,324) a seis meses. A
   tres meses el encadenado sí gana (0,220 frente a 0,244), pero el margen es estrecho. La razón
   es que los errores de los canales se compensan parcialmente al sumarlos, así que el agregado
   resulta más fácil que sus partes. Conviene enseñarlo precisamente porque contradice la
   intuición de "más detalle, mejor modelo".
2. **El ingreso es bastante más difícil de pronosticar que los suscriptores.** MAPE del 11,3%
   frente al 4,2% de la página anterior, con el mismo esfuerzo de modelado. Tiene sentido: la
   base de suscriptores es contractual y cambia despacio; el ingreso mezcla esa base con compra
   discrecional.
3. **Y el ranking de dificultad por canal lo confirma**, ordenando los cuatro justo por cuánto
   decide el cliente en cada compra: suscripción (MASE 0,29) → boutique (0,50) → máquina (0,69) →
   **tienda online (0,80), a un paso de no batir al naive estacional**. Cuanto más discrecional
   es la compra, menos predecible el ingreso.

Los cuatro modelos, eso sí, dejan muy atrás al naive estacional (MASE 1,45), que con una serie
que casi duplica cada año no tiene ninguna posibilidad.

### ¿Cuánto aporta encadenar con el modelo de cohortes?

El canal de suscripción es un tercio del ingreso y tiene una ventaja que los otros no: su base
es contractual y se puede pronosticar con la mecánica de cohortes de la página anterior, que
sabe quién está de pausa y cuándo vuelve. La pregunta es si ese conocimiento sobrevive al
multiplicarlo por el ingreso por activo.

In [15]:
def subscription_chained(train, horizon):
    cutoff = train.index[-1]
    subscribers = us.forecast_active_subscribers(facts, cutoff, horizon, apply_pause=False)
    arpu = sarima_forecast(revenue_per_active.reindex(train.index), horizon)["yhat"].to_numpy()
    return subscribers * arpu
subscription_chained.__name__ = "cohortes_x_ingreso_por_activo"

subscription_models = {
    "sarima_airline": sarima_forecast,
    "cohortes_x_ingreso_por_activo": subscription_chained,
}
subscription_backtests = {
    name: ts.walk_forward_backtest(channels["suscripcion"], fn, horizon=FORECAST_HORIZON,
                                   n_folds=BACKTEST_FOLDS, step=1, season_length=SEASON_LENGTH,
                                   model_name=name, on_error="skip")
    for name, fn in subscription_models.items()
}
print("Sólo el canal de suscripción:")
print(ts.compare_backtests(subscription_backtests)[["model", "mae", "mape", "mase", "bias"]]
      .round(3).to_string(index=False))
print()
cv = revenue_per_active.std() / revenue_per_active.mean()
print(f"Ingreso reconocido por activo a cierre: media {revenue_per_active.mean():.2f} €, "
      f"coeficiente de variación {cv:.1%}")

Sólo el canal de suscripción:
                        model      mae   mape  mase     bias
cohortes_x_ingreso_por_activo 4805.728  6.509 0.190 1349.913
               sarima_airline 7154.478 10.100 0.285 4199.650

Ingreso reconocido por activo a cierre: media 24.95 €, coeficiente de variación 13.4%


**Encadenar funciona donde se aplica.** En el canal de suscripción el MASE baja de 0,287 a
0,186, un 35% menos de error, y el MAPE del 10,2% al 6,4%. Tiene sentido: el ingreso por activo
es muy estable (coeficiente de variación del 13%), así que el problema se reduce a pronosticar
la base de suscriptores, que es exactamente lo que el modelo de cohortes hace bien.

**Pero en el agregado casi no se nota** (MASE 0,329 frente a 0,324 del SARIMA directo a seis
meses, y 0,219 frente a 0,244 a tres). La razón es la misma que ya apareció con el bottom-up:
el error del total está dominado por los canales discrecionales, y mejorar el canal más
predecible mueve poco la aguja. Es un recordatorio útil de que optimizar la parte fácil de un
agregado rinde poco.

### El intervalo de la suma de canales

Esta era la deuda que quedaba abierta: la suma de cuatro SARIMA no tiene intervalo propio, y
componerlo a partir de los cuatro intervalos individuales parecía exigir convolucionar cuatro
distribuciones correlacionadas entre sí.

Conviene comprobar ese supuesto antes de darlo por bueno.

In [16]:
# Errores de cada canal alineados por (pliegue, horizonte): así se pueden correlacionar.
channel_errors = pd.DataFrame({
    name: result.predictions.set_index(["fold", "h"])["error"]
    for name, result in per_channel.items()
})
correlation = channel_errors.corr()
off_diagonal = correlation.to_numpy()[np.triu_indices(len(channels), 1)]
print("Correlación entre los errores de los cuatro canales:")
print(correlation.round(3).to_string())
print()
print(f"Correlación media fuera de la diagonal: {off_diagonal.mean():.3f}")

var_independent = channel_errors.var(ddof=1).sum()
var_actual = channel_errors.sum(axis=1).var(ddof=1)
print()
print("Desviación típica del error de la SUMA:")
print(f"  suponiendo independencia : {np.sqrt(var_independent):,.0f} €")
print(f"  medida directamente      : {np.sqrt(var_actual):,.0f} €")
print(f"  diferencia               : {(np.sqrt(var_independent / var_actual) - 1) * 100:+.0f}%")

Correlación entre los errores de los cuatro canales:
             suscripcion  online  tienda  maquina
suscripcion        1.000   0.066   0.408   -0.063
online             0.066   1.000  -0.298   -0.132
tienda             0.408  -0.298   1.000    0.224
maquina           -0.063  -0.132   0.224    1.000

Correlación media fuera de la diagonal: 0.034

Desviación típica del error de la SUMA:
  suponiendo independencia : 21,773 €
  medida directamente      : 21,114 €
  diferencia               : +3%


**El supuesto era falso.** Los errores de los cuatro canales están prácticamente **incorrelados**
(correlación media 0,03), así que suponer independencia no infla ni desinfla la banda: la dejaría
un 3% **más ancha** de lo debido, es decir, ligeramente conservadora, no optimista.

Hay correlaciones apreciables por pares —suscripción con boutique +0,41, online con boutique
−0,30— pero de signos opuestos, y al sumar los cuatro canales se cancelan. Es un buen recordatorio
de que "estas series están correlacionadas" es una intuición que hay que medir antes de dejar que
bloquee una decisión de diseño.

En cualquier caso, la vía correcta no es componer los intervalos individuales sino **medir
directamente el error de la suma en el backtesting**, que es lo que hace `empirical_interval`: los
errores del bottom-up ya incorporan la correlación que haya, sea la que sea, sin suponer nada.

## 7. Forecast a 6 meses

In [17]:
# Intervalos empíricos para los modelos que no los producen, a partir de sus errores
# walk-forward. Misma técnica que en la página 2: el sesgo se estima por horizonte y la
# dispersión se suaviza con sd(h) = a*sqrt(h) para no depender de cinco puntos por horizonte.
empirical_bands = {
    name: ts.empirical_interval(result.predictions, level=INTERVAL_LEVEL)
    for name, result in backtests.items() if name != "naive_estacional"
}
print("Calibración empírica del modelo encadenado:")
print(empirical_bands["encadenado_cohortes"][
    ["mean_rel_error", "sd_raw", "sd_smoothed", "calibration_factor"]].round(4).to_string())

final_total = sarima_forecast(total_revenue, FORECAST_HORIZON)
future_months = pd.date_range(total_revenue.index[-1] + pd.offsets.MonthBegin(1),
                              periods=FORECAST_HORIZON, freq="MS")
final_total.index = future_months

channel_forecasts = {name: sarima_forecast(s, FORECAST_HORIZON).set_index(future_months)
                     for name, s in channels.items()}
bottom_up_raw = sum(f.yhat for f in channel_forecasts.values())
bottom_up_band = empirical_bands["bottom_up_canal"]
bottom_up_total = bottom_up_raw * bottom_up_band["calibration_factor"].to_numpy()
bottom_up_lower = bottom_up_raw * bottom_up_band["lower_factor"].to_numpy()
bottom_up_upper = bottom_up_raw * bottom_up_band["upper_factor"].to_numpy()
chained_raw = chained_forecast(total_revenue, FORECAST_HORIZON)
chained_band = empirical_bands["encadenado_cohortes"]
chained_point = chained_raw * chained_band["calibration_factor"].to_numpy()
chained_lower = chained_raw * chained_band["lower_factor"].to_numpy()
chained_upper = chained_raw * chained_band["upper_factor"].to_numpy()

forecast_table = pd.DataFrame({
    "total_sarima": final_total.yhat.round(0),
    "total_lo": final_total.yhat_lower.round(0),
    "total_hi": final_total.yhat_upper.round(0),
    "suma_canales": bottom_up_total.round(0),
    "encadenado": chained_point.round(0),
    "encadenado_lo": chained_lower.round(0),
    "encadenado_hi": chained_upper.round(0),
    **{CHANNEL_LABEL[n]: f.yhat.round(0) for n, f in channel_forecasts.items()},
})
print(f"Diferencia entre el total top-down y la suma de canales: "
      f"{(bottom_up_total.sum() / final_total.yhat.sum() - 1):+.1%} sobre los {FORECAST_HORIZON} meses")
forecast_table

Calibración empírica del modelo encadenado:
   mean_rel_error  sd_raw  sd_smoothed  calibration_factor
h                                                         
1          0.0212  0.1021       0.0493              0.9793
2          0.0830  0.0286       0.0698              0.9234
3          0.1074  0.1066       0.0854              0.9030
4          0.1368  0.0569       0.0987              0.8796
5          0.1200  0.0764       0.1103              0.8929
6          0.1500  0.0994       0.1208              0.8696


Diferencia entre el total top-down y la suma de canales: -2.2% sobre los 6 meses


,total_sarima,total_lo,total_hi,suma_canales,encadenado,encadenado_lo,encadenado_hi,Suscripción,Tienda online,Boutique,Máquinas
2026-09-01,266308.0,239526.0,296084.0,267062.0,270953.0,253824.0,288082.0,65516.0,101978.0,50931.0,55043.0
2026-10-01,336893.0,299371.0,379117.0,332763.0,331136.0,301531.0,360740.0,79514.0,145172.0,71495.0,65843.0
2026-11-01,360159.0,316558.0,409766.0,351156.0,352302.0,313726.0,390877.0,81470.0,165109.0,80366.0,64143.0
2026-12-01,375969.0,327133.0,432096.0,361011.0,362308.0,316500.0,408117.0,93362.0,179174.0,75022.0,68464.0
2027-01-01,376479.0,324506.0,436775.0,370226.0,373477.0,320683.0,426271.0,96506.0,194004.0,70235.0,61960.0
2027-02-01,334049.0,285396.0,390995.0,323261.0,327990.0,277201.0,378780.0,96249.0,171456.0,62551.0,51989.0


In [18]:
fig = go.Figure()
fig.add_trace(go.Scatter(x=total_revenue.index, y=total_revenue.to_numpy(), name="Histórico",
                         mode="lines", line=dict(color=C_INK, width=2.5)))
fig.add_trace(go.Scatter(x=[*future_months, *future_months[::-1]],
                         y=[*final_total.yhat_upper, *final_total.yhat_lower[::-1]],
                         fill="toself", fillcolor="rgba(42,120,214,0.14)", line=dict(width=0),
                         name="Banda 80%", hoverinfo="skip"))
bridge_x = [total_revenue.index[-1], *future_months]
fig.add_trace(go.Scatter(x=bridge_x, y=[float(total_revenue.iloc[-1]), *final_total.yhat],
                         name="SARIMA airline", mode="lines+markers",
                         line=dict(color=C_BLUE, width=2.5), marker=dict(size=8)))
fig.add_trace(go.Scatter(x=bridge_x, y=[float(total_revenue.iloc[-1]), *bottom_up_total],
                         name="Suma de canales", mode="lines+markers",
                         line=dict(color=C_ORANGE, width=2, dash="dash"), marker=dict(size=7)))
fig.add_vline(x=total_revenue.index[-1], line=dict(color=C_MUTED, width=1, dash="dot"))
fig.update_layout(**{**PLOT_LAYOUT, "height": 470},
                  title=f"Ingreso total: forecast a {FORECAST_HORIZON} meses",
                  yaxis_title="€ / mes")
fig.show()

fig = go.Figure()
for name, s in channels.items():
    fig.add_trace(go.Scatter(x=s.index, y=s.to_numpy(), name=CHANNEL_LABEL[name], mode="lines",
                             line=dict(color=CHANNEL_COLOR[name], width=2)))
    f = channel_forecasts[name]
    fig.add_trace(go.Scatter(x=[s.index[-1], *future_months],
                             y=[float(s.iloc[-1]), *f.yhat], showlegend=False, mode="lines",
                             line=dict(color=CHANNEL_COLOR[name], width=2, dash="dot")))
fig.add_vline(x=total_revenue.index[-1], line=dict(color=C_MUTED, width=1, dash="dot"))
fig.update_layout(**{**PLOT_LAYOUT, "height": 460}, title="Forecast por canal",
                  yaxis_title="€ / mes")
fig.show()

## 8. Volcado a `analysis/outputs/ingresos.json`

In [19]:
def records(series_obj, key="value"):
    return ts.series_to_records(series_obj, value_key=key)

payload = {
    "meta": {
        "page": "03_ingresos",
        "title": "Serie temporal: ingresos totales",
        "generated_at": datetime.now(timezone.utc).isoformat(timespec="seconds"),
        "source_tables": ["fct_subscriptions_monthly", "fct_shop_orders"],
        "grain": "mensual (diario en la parte de tienda)",
        "history_start": total_revenue.index.min().strftime("%Y-%m-%d"),
        "history_end": total_revenue.index.max().strftime("%Y-%m-%d"),
        "n_months": int(len(total_revenue)),
        "forecast_horizon": FORECAST_HORIZON,
        "interval_level": INTERVAL_LEVEL,
        "revenue_measure": "recognized_revenue_eur",
        "revenue_measure_note": (
            "De las tres medidas del mart se usa el ingreso reconocido: es la única devengada. "
            "La caja cobrada acusa el hueco de la migración de pasarela y el MRR contratado es "
            "una tarifa vigente, no un ingreso."
        ),
        "scope_note": (
            "Ingresos = suscripción + tienda online + boutique + máquinas, los cuatro canales "
            "del negocio. La máquina entró al crear el mart fct_machine_orders y pesa un 21,5%."
        ),
        "total_revenue_eur": float(total_revenue.sum()),
    },
    "series": {
        "total": records(total_revenue, "eur"),
        **{name: records(s, "eur") for name, s in channels.items()},
        "shop_daily": records(shop_total_daily, "eur"),
    },
    "channel_mix": {
        "share_pct": {name: float(share[name]) for name in channels},
        "by_month_pct": [
            {"date": d.strftime("%Y-%m-%d"), **{n: float(mix_pct.loc[d, n]) for n in channels}}
            for d in mix_pct.index
        ],
    },
    "payment_gap": {
        "imperfection": "Migración de pasarela de pago",
        "detected_month": GAP_MONTH.strftime("%Y-%m-%d"),
        "detection_method": ("Ratio caja / MRR contratado. El denominador no crece con el "
                             "negocio ni depende de los días del mes, así que aísla el fallo de "
                             "registro; los febreros cortos dejan de ser un distractor."),
        "cash_ratio_median": float(mediana),
        "cash_ratio_at_gap": float(gap.loc[GAP_MONTH, "cash_ratio"]),
        "drop_vs_median_pct": float(caida * 100),
        "cash_ratio": records(pd.Series(gap.cash_ratio.to_numpy(), index=gap.index), "ratio"),
        "collected_per_active": records(
            pd.Series(gap.collected_per_active.to_numpy(), index=gap.index), "eur"),
        "mrr_per_active": records(
            pd.Series(gap.mrr_per_active.to_numpy(), index=gap.index), "eur"),
    },
    "decomposition": decomposition.to_dict(),
    "channel_decomposition_strength": {
        name: {"trend": float(d.trend_strength),
               "seasonal": float(d.seasonal_strength[SEASON_LENGTH])}
        for name, d in decompositions.items()
    },
    "seasonal_profile_by_channel_note": (
        "Mediana de los últimos 24 meses. En el año de arranque la tendencia es tan pequeña "
        "que el porcentaje sobre tendencia se dispara y no es interpretable."),
    "seasonal_profile_by_channel": [
        {"month": int(i + 1), **{n: float(profile_frame.iloc[i][n]) for n in channels}}
        for i in range(12)
    ],
    "weekly_seasonality": {
        "scope": "sólo tienda (online + boutique); la suscripción no tiene grano diario",
        "periods": list(daily_decomposition.periods),
        "seasonal_strength": {str(p): float(v)
                              for p, v in daily_decomposition.seasonal_strength.items()},
        "trend_strength": float(daily_decomposition.trend_strength),
        "weekend_ratio": weekend_ratio,
        "by_weekday": [
            {"weekday": i, "label": DAYS[i],
             "mean_eur_total": float(dow_frame.total.iloc[i]),
             "mean_eur_online": float(dow_frame["online"].iloc[i]),
             "mean_eur_store": float(dow_frame["tienda"].iloc[i]),
             "mstl_component": float(weekly_profile.iloc[i])}
            for i in range(7)
        ],
    },
    "stationarity": {"level": report_level.to_dict(), "log": report_log.to_dict()},
    "backtest": {
        "horizon": FORECAST_HORIZON,
        "n_folds": BACKTEST_FOLDS,
        "window": "expanding",
        "leaderboard": json.loads(leaderboard.to_json(orient="records")),
        "models": {name: r.to_dict() for name, r in backtests.items()},
        "short_horizon_leaderboard": json.loads(
            ts.compare_backtests(short).to_json(orient="records")),
        "per_channel": json.loads(per_channel_table.to_json(orient="records")),
        "subscription_channel": json.loads(
            ts.compare_backtests(subscription_backtests).to_json(orient="records")),
        "channel_error_correlation": json.loads(correlation.to_json(orient="index")),
        "sum_interval_check": {
            "mean_off_diagonal_correlation": float(off_diagonal.mean()),
            "sd_if_independent_eur": float(np.sqrt(var_independent)),
            "sd_measured_eur": float(np.sqrt(var_actual)),
            "note": ("Los errores de canal están casi incorrelados, así que suponer "
                     "independencia daría una banda un 3% más ancha, no más estrecha. "
                     "El intervalo publicado mide directamente el error de la suma."),
        },
        "empirical_calibration": {
            name: json.loads(band.reset_index().to_json(orient="records"))
            for name, band in empirical_bands.items()
        },
    },
    "forecast": {
        "horizon": FORECAST_HORIZON,
        "months": [d.strftime("%Y-%m-%d") for d in future_months],
        "models": {
            "sarima_airline": {
                "label": "SARIMA(0,1,1)(0,1,1)12 sobre log del agregado",
                "yhat": [float(v) for v in final_total.yhat],
                "yhat_lower": [float(v) for v in final_total.yhat_lower],
                "yhat_upper": [float(v) for v in final_total.yhat_upper],
                "interval_source": "analítico del modelo",
            },
            "bottom_up_canal": {
                "label": "Suma de un SARIMA por canal",
                "yhat": [float(v) for v in bottom_up_total],
                "yhat_raw": [float(v) for v in bottom_up_raw],
                "yhat_lower": [float(v) for v in bottom_up_lower],
                "yhat_upper": [float(v) for v in bottom_up_upper],
                "interval_source": "empírico, del error medido de la suma en el backtesting",
            },
            "encadenado_cohortes": {
                "label": "Cohortes x ingreso por activo + SARIMA en los otros tres canales",
                "yhat": [float(v) for v in chained_point],
                "yhat_raw": [float(v) for v in chained_raw],
                "yhat_lower": [float(v) for v in chained_lower],
                "yhat_upper": [float(v) for v in chained_upper],
                "interval_source": "empírico, de los errores walk-forward por horizonte",
            },
        },
        "by_channel": {
            name: {"yhat": [float(v) for v in f.yhat],
                   "yhat_lower": [float(v) for v in f.yhat_lower],
                   "yhat_upper": [float(v) for v in f.yhat_upper]}
            for name, f in channel_forecasts.items()
        },
    },
    "insights": [
        ("De las tres medidas de ingreso sólo la caja acusa la migración de pasarela: en marzo "
         "de 2025 el ratio caja/MRR cae a 0,819 frente a 1,02 de mediana, un 20% por debajo y "
         "mínimo absoluto del histórico, con el MRR contratado plano."),
        ("Ningún canal domina el ingreso: 33,5% suscripción, 25,2% tienda online, 21,5% "
         "máquinas y 19,8% boutique. La máquina pesa más de lo que sugería el agregado por "
         "cliente porque incluye 616 pedidos de boutique sin fidelización."),
        ("La boutique es el canal más estacional (F_S = 0,94) y la suscripción el menos (0,77), "
         "y además van desfasados: la tienda hace techo en octubre-noviembre y la suscripción "
         "en marzo. El agregado mezcla estacionalidades distintas."),
        ("La estacionalidad semanal pesa tres veces más en el ingreso de tienda (F_S = 0,65) que "
         "en las altas de suscripción (0,21): el fin de semana factura un 30% menos."),
        ("Desagregar NO mejora el forecast del agregado: ni por canal (MASE 0,375) ni "
         "encadenando cohortes (0,329) se bate al SARIMA directo (0,324). Los errores de los "
         "canales se compensan al sumarlos."),
        ("Encadenar sí funciona donde se aplica: en el canal de suscripción el error cae un 35% "
         "(MASE 0,29 a 0,19) al pronosticarlo como activos x ingreso por activo."),
        ("El ingreso es mucho más difícil de pronosticar que los suscriptores (MAPE 11,3% frente "
         "a 4,2%), y la dificultad ordena los canales por discrecionalidad de la compra: "
         "suscripción 0,29, boutique 0,50, máquina 0,69, tienda online 0,80."),
    ],
}

with open(OUTPUT_PATH, "w", encoding="utf-8") as handle:
    json.dump(payload, handle, ensure_ascii=False, indent=2)
print(f"Guardado {OUTPUT_PATH.relative_to(PROJECT_ROOT)} "
      f"({OUTPUT_PATH.stat().st_size / 1024:.0f} KB)")
print("Claves de primer nivel:", list(payload))

Guardado analysis\outputs\ingresos.json (221 KB)
Claves de primer nivel: ['meta', 'series', 'channel_mix', 'payment_gap', 'decomposition', 'channel_decomposition_strength', 'seasonal_profile_by_channel_note', 'seasonal_profile_by_channel', 'weekly_seasonality', 'stationarity', 'backtest', 'forecast', 'insights']


In [20]:
with open(OUTPUT_PATH, encoding="utf-8") as handle:
    reloaded = json.load(handle)

checks = {
    "serie total": len(reloaded["series"]["total"]) == len(total_revenue),
    "cuatro canales": all(len(reloaded["series"][n]) == len(total_revenue) for n in channels),
    "serie diaria": len(reloaded["series"]["shop_daily"]) == len(shop_total_daily),
    "hueco de pasarela": reloaded["payment_gap"]["detected_month"] == "2025-03-01",
    "serie del ratio": len(reloaded["payment_gap"]["cash_ratio"]) == len(total_revenue),
    "descomposición": len(reloaded["decomposition"]["trend"]) == len(total_revenue),
    "estacionalidad semanal": len(reloaded["weekly_seasonality"]["by_weekday"]) == 7,
    "leaderboard": len(reloaded["backtest"]["leaderboard"]) == len(models),
    "canal de suscripción": len(reloaded["backtest"]["subscription_channel"]) == 2,
    "intervalos del encadenado": (
        len(reloaded["forecast"]["models"]["encadenado_cohortes"]["yhat_lower"])
        == FORECAST_HORIZON),
    "forecast e intervalos": (len(reloaded["forecast"]["models"]["sarima_airline"]["yhat"])
                              == FORECAST_HORIZON
                              and len(reloaded["forecast"]["models"]["sarima_airline"]["yhat_lower"])
                              == FORECAST_HORIZON),
    "forecast por canal": all(len(v["yhat"]) == FORECAST_HORIZON
                              for v in reloaded["forecast"]["by_channel"].values()),
}
for label, ok in checks.items():
    print(f"  {'OK ' if ok else 'FALLO'} {label}")
assert all(checks.values()), "El JSON de salida no tiene la forma esperada."
print()
print("JSON verificado.")

  OK  serie total
  OK  cuatro canales
  OK  serie diaria
  OK  hueco de pasarela
  OK  serie del ratio
  OK  descomposición
  OK  estacionalidad semanal
  OK  leaderboard
  OK  canal de suscripción
  OK  intervalos del encadenado
  OK  forecast e intervalos
  OK  forecast por canal

JSON verificado.


## Conclusiones

1. **Elegir la medida de ingreso es la decisión que más condiciona esta página.** Las tres
   medidas del mart cuentan historias distintas del mismo mes. El ingreso reconocido es el único
   que representa servicio prestado, y es el único inmune al hueco de la pasarela.
2. **La imperfección se encuentra eligiendo bien el denominador.** En valor absoluto el bache de
   marzo de 2025 desaparece dentro del crecimiento, y por suscriptor activo todavía compite con
   los febreros cortos. Dividiendo la caja entre el MRR contratado —un denominador que no crece
   ni depende de los días del mes— queda como mínimo absoluto del histórico y sin ambigüedad.
3. **Más detalle no es mejor modelo — en el agregado.** Ni sumar cuatro SARIMA por canal ni
   encadenar el canal de suscripción con el modelo de cohortes baten al modelo directo sobre el
   total a seis meses. Es el hallazgo metodológico de la página, y tiene un corolario práctico:
   mejorar el canal más predecible mueve poco un agregado cuyo error lo dominan los canales
   discrecionales.
4. **Pero encadenar sí funciona donde se aplica.** En el propio canal de suscripción el error cae
   un 35% (MASE 0,29 → 0,19). Si lo que se quiere planificar es la caja recurrente y no el total,
   ese es el modelo a usar.
5. **La predictibilidad decrece con la discrecionalidad de la compra:** suscripción (MASE 0,29) →
   boutique (0,50) → máquina (0,69) → tienda online (0,80). Es una jerarquía con sentido de
   negocio, no un accidente del ajuste.
6. **La estacionalidad semanal es real y grande en tienda**, aunque se promedie al agregar a mes.
   Importa para operaciones, no para el forecast mensual.

### Deuda técnica de esta página

Ninguna abierta en esta versión.

La que quedaba —el intervalo de la suma de canales— estaba mal planteada: daba por hecho que los
errores de los canales estaban correlacionados y que había que convolucionar sus distribuciones.
Medida, la correlación media resulta ser 0,03, y el intervalo correcto sale de medir el error de
la suma directamente en el backtesting, sin suponer nada sobre sus componentes. Los tres modelos
publican ya su banda del 80%.

Resueltas en esta versión: la venta de máquinas ya está dentro del ingreso total (mart
`fct_machine_orders`, y resultan ser 840.859 € — un 21,5%, más de lo que sugería el agregado por
cliente, porque incluye los 616 pedidos de boutique sin fidelización), y el encadenado con el
modelo de cohortes ya está implementado y medido, sin acoplar los notebooks: la mecánica vive en
`analysis/utils_subscriptions.py` y ambos la importan.